# Method Gating for Ensemble Voting

**Problem:** Some similarity measures perform poorly on certain constraint patterns — their top-ranked genes don't actually follow the pattern shape. We need a way to automatically silence those methods before they vote.

**Approach:**
1. **Act 1:** Try the professor's suggestion — hardcoded per-metric score thresholds. Show why fixed thresholds don't generalize across clusters.
2. **Act 2:** Test data-driven gates that evaluate whether a method's top genes actually match the pattern. Compare three candidates, pick the best one.

**Data:** Log-transformed (LT) only. 6 methods × 4 clusters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats as scipy_stats
import os, pickle

%matplotlib inline

# Load pre-computed metric results from last week
results = pickle.load(open('../Week of April 9th/metrics_results.pkl', 'rb'))
constraint_patternsLT = results['constraint_patternsLT']
all_methodsLT = results['all_methodsLT']
method_names = list(all_methodsLT.keys())

# Load gene expression data (LT)
x = [0, 3, 6, 9]
gene_dataLT = {}
for cluster_name in constraint_patternsLT:
    gene_file = f'../analysis data/gene_countsLT/{cluster_name}_annotated.csv'
    df = pd.read_csv(gene_file, index_col=0)
    df.columns = x
    gene_dataLT[cluster_name] = df

cluster_names = list(constraint_patternsLT.keys())
print(f'Loaded {len(method_names)} methods: {", ".join(method_names)}')
print(f'Clusters: {cluster_names}')
print(f'Genes per cluster: {gene_dataLT[cluster_names[0]].shape[0]}')

In [ ]:
# Helper: get a method's top-N genes for a cluster
def get_top_genes(method_name, cluster_name, n=20):
    """Return top-N gene names for a method on a cluster."""
    scores, ascending = all_methodsLT[method_name]
    s = scores[cluster_name]['constraint']
    if ascending:
        return s.nsmallest(n).index.tolist()
    else:
        return s.nlargest(n).index.tolist()

# Helper: get a method's top-N raw scores
def get_top_scores(method_name, cluster_name, n=20):
    """Return top-N scores (Series) for a method on a cluster."""
    scores, ascending = all_methodsLT[method_name]
    s = scores[cluster_name]['constraint']
    return s.nsmallest(n) if ascending else s.nlargest(n)

# Helper: plot gene curves + pattern for a method on a cluster
def plot_method_cluster(method_name, cluster_name, ax, n=20, title=None, alpha=0.4):
    """Plot top-N gene curves and constraint pattern on given axes."""
    top_genes = get_top_genes(method_name, cluster_name, n)
    gene_df = gene_dataLT[cluster_name]
    pattern_vals = constraint_patternsLT[cluster_name]['constraint']
    
    plotted = 0
    for gene in top_genes:
        if gene in gene_df.index:
            ax.plot(x, gene_df.loc[gene], color='steelblue', alpha=alpha, linewidth=1)
            plotted += 1
    ax.plot(x, pattern_vals, color='crimson', linewidth=2.5, linestyle='--')
    ax.set_xlabel('Timepoint')
    ax.set_ylabel('Gene counts')
    ax.set_xticks(x)
    if title:
        ax.set_title(title, fontsize=10)
    legend_elements = [
        Line2D([0], [0], color='crimson', linewidth=2.5, linestyle='--', label='Constraint pattern'),
        Line2D([0], [0], color='steelblue', alpha=alpha, label=f'Top {n} genes (n={plotted})'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=8)

---
## Act 1: Hardcoded Per-Metric Thresholds

The intuitive approach: for each metric, set a quality floor on the raw score. If a method's best genes don't meet that floor on a cluster, the method doesn't vote there.

In [ ]:
# Fixed thresholds: a method's top-1 gene must meet this bar to vote
# For similarity metrics (higher=better): score must be ABOVE threshold
# For distance metrics (lower=better): score must be BELOW threshold
FIXED_THRESHOLDS = {
    'Pearson':  ('>', 0.99),
    'DTW':      ('<', 0.08),
    'Cosine':   ('>', 0.999),
    'Frechet':  ('<', 0.03),
    'MSE':      ('<', 0.001),
    'sMAPE':    ('<', 0.10),
}

def apply_fixed_thresholds(thresholds, n=20):
    """Check which methods pass fixed thresholds per cluster.
    Returns DataFrame: methods x clusters, values = # qualifying genes."""
    grid = pd.DataFrame(index=method_names, columns=cluster_names, dtype=int)
    
    for method_name in method_names:
        op, thresh = thresholds[method_name]
        scores_dict, ascending = all_methodsLT[method_name]
        for cluster_name in cluster_names:
            s = scores_dict[cluster_name]['constraint']
            if op == '>':
                count = (s > thresh).sum()
            else:
                count = (s < thresh).sum()
            grid.loc[method_name, cluster_name] = count
    return grid

qualifying = apply_fixed_thresholds(FIXED_THRESHOLDS)

# Show active/silenced grid
active_grid = qualifying.map(lambda x: '✓' if x > 0 else '✗')
print("Active/Silenced Grid (fixed thresholds):")
print(active_grid)
print()
print("Qualifying gene counts:")
print(qualifying)

In [ ]:
# Threshold sweep: show that no single per-method threshold works across all clusters.
# For each method, sweep threshold and show how many clusters it's active on.

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, method_name in enumerate(method_names):
    ax = axes[idx]
    scores_dict, ascending = all_methodsLT[method_name]
    
    # Get the best (top-1) score per cluster
    best_scores = {}
    for cluster_name in cluster_names:
        s = scores_dict[cluster_name]['constraint']
        best = s.min() if ascending else s.max()
        best_scores[cluster_name] = best
    
    # Get the 20th-best score per cluster (the threshold where top-20 starts to fail)
    score_at_20 = {}
    for cluster_name in cluster_names:
        s = scores_dict[cluster_name]['constraint']
        sorted_s = s.sort_values(ascending=ascending)
        score_at_20[cluster_name] = sorted_s.iloc[19] if len(sorted_s) >= 20 else sorted_s.iloc[-1]
    
    # Plot: for each cluster, show the range of top-20 scores
    cluster_labels = [c.replace('LT', '') for c in cluster_names]
    positions = range(len(cluster_names))
    
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
    for i, cluster_name in enumerate(cluster_names):
        s = scores_dict[cluster_name]['constraint']
        top20 = s.nsmallest(20) if ascending else s.nlargest(20)
        ax.scatter([i]*20, top20.values, alpha=0.3, color=colors[i], s=15)
        ax.scatter([i], [best_scores[cluster_name]], color=colors[i], s=80, 
                   zorder=5, edgecolors='black', linewidth=1)
    
    # Draw the fixed threshold line
    _, thresh = FIXED_THRESHOLDS[method_name]
    ax.axhline(y=thresh, color='red', linestyle='--', linewidth=2, label=f'Threshold = {thresh}')
    
    ax.set_xticks(positions)
    ax.set_xticklabels(cluster_labels, fontsize=8)
    ax.set_title(method_name, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle('Fixed Thresholds: Top-20 Score Distributions per Cluster\n'
             '(Large dots = best score, small dots = top-20 genes)', fontsize=12)
plt.tight_layout()
plt.savefig('plots/act1_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print("Any single threshold line either silences good clusters or permits bad ones.")

### What if we manually picked thresholds to silence everything except sMAPE on clusterThree?

For each method, we need a threshold tight enough that the method's clusterThree scores fail — but what does that do to the other clusters?

In [ ]:
# For each non-sMAPE method, find the threshold that would silence it on clusterThree.
# Then show what that threshold does to ALL clusters.

print("=" * 70)
print("EXPERIMENT: Manually pick thresholds to get sMAPE-only on clusterThree")
print("=" * 70)

# For each method, the threshold must be tighter than the BEST score on clusterThree
# (i.e., even the best gene on clusterThree doesn't pass)
forced_thresholds = {}
c3 = 'clusterThreeLT'

for method_name in method_names:
    if method_name == 'sMAPE':
        # Keep sMAPE's threshold loose so it still works on clusterThree
        forced_thresholds[method_name] = FIXED_THRESHOLDS[method_name]
        continue
    
    scores_dict, ascending = all_methodsLT[method_name]
    s = scores_dict[c3]['constraint']
    
    if ascending:  # lower is better — need threshold BELOW the best (lowest) score on c3
        best_c3 = s.min()
        forced_thresh = best_c3 * 0.99  # just below the best score
        forced_thresholds[method_name] = ('<', forced_thresh)
    else:  # higher is better — need threshold ABOVE the best (highest) score on c3
        best_c3 = s.max()
        forced_thresh = best_c3 * 1.001  # just above the best score
        forced_thresholds[method_name] = ('>', forced_thresh)

# Show forced thresholds vs original
print("\nForced thresholds (to silence on clusterThree):")
print(f"  {'Method':<10} {'Original':<20} {'Forced':<25} {'Change'}")
print(f"  {'-'*70}")
for m in method_names:
    orig_op, orig_t = FIXED_THRESHOLDS[m]
    forced_op, forced_t = forced_thresholds[m]
    if m == 'sMAPE':
        print(f"  {m:<10} {orig_op} {orig_t:<17} {forced_op} {forced_t:<22.6f} (unchanged)")
    else:
        print(f"  {m:<10} {orig_op} {orig_t:<17} {forced_op} {forced_t:<22.6f} ← TIGHTENED")

# Apply forced thresholds
forced_qualifying = apply_fixed_thresholds(forced_thresholds)
forced_active = forced_qualifying.map(lambda x: '✓' if x > 0 else '✗')

print(f"\nActive/Silenced Grid with FORCED thresholds:")
print(forced_active)
print()
print("Qualifying gene counts:")
print(forced_qualifying)

# Show collateral damage
print("\n" + "=" * 70)
print("COLLATERAL DAMAGE: Methods silenced on OTHER clusters")
print("=" * 70)
for cluster_name in cluster_names:
    short = cluster_name.replace('LT', '')
    orig_active = sum(1 for m in method_names 
                      if apply_fixed_thresholds(FIXED_THRESHOLDS).loc[m, cluster_name] > 0)
    forced_active_count = sum(1 for m in method_names 
                              if forced_qualifying.loc[m, cluster_name] > 0)
    lost = orig_active - forced_active_count
    print(f"  {short}: {orig_active} → {forced_active_count} active methods ({lost} lost)")

In [ ]:
# Visualize: same threshold sweep plot but with BOTH threshold lines
# Original (red dashed) vs Forced (black dashed)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, method_name in enumerate(method_names):
    ax = axes[idx]
    scores_dict, ascending = all_methodsLT[method_name]
    
    cluster_labels = [c.replace('LT', '') for c in cluster_names]
    positions = range(len(cluster_names))
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
    
    for i, cluster_name in enumerate(cluster_names):
        s = scores_dict[cluster_name]['constraint']
        top20 = s.nsmallest(20) if ascending else s.nlargest(20)
        ax.scatter([i]*20, top20.values, alpha=0.3, color=colors[i], s=15)
        best = s.min() if ascending else s.max()
        ax.scatter([i], [best], color=colors[i], s=80, zorder=5, edgecolors='black', linewidth=1)
    
    # Original threshold
    _, orig_thresh = FIXED_THRESHOLDS[method_name]
    ax.axhline(y=orig_thresh, color='red', linestyle='--', linewidth=1.5, alpha=0.5,
               label=f'Original = {orig_thresh}')
    
    # Forced threshold
    _, forced_thresh = forced_thresholds[method_name]
    if method_name != 'sMAPE':
        ax.axhline(y=forced_thresh, color='black', linestyle='-', linewidth=2.5,
                   label=f'Forced = {forced_thresh:.4f}')
    
    ax.set_xticks(positions)
    ax.set_xticklabels(cluster_labels, fontsize=8)
    ax.set_title(method_name, fontweight='bold')
    ax.legend(fontsize=6, loc='best')

plt.suptitle('Fixed Thresholds: Original (red) vs Forced to silence clusterThree (black)\n'
             'Notice how the forced thresholds kill methods on OTHER clusters too',
             fontsize=11)
plt.tight_layout()
plt.savefig('plots/act1_forced_thresholds.png', dpi=150, bbox_inches='tight')
plt.show()
print("The black lines show what it would take to silence non-sMAPE methods on clusterThree.")
print("They cut through the score distributions of other clusters where those methods work fine.")

**Conclusion from Act 1:** Fixed per-metric thresholds are inherently fragile. The same method has very different score distributions across clusters (different patterns have different scales, shapes, and difficulty). A threshold tuned to silence MSE on clusterThree will also silence MSE on clusterOne where it works fine.

We need a gate that evaluates the **output quality** — do the top genes actually match the pattern? — rather than the raw score values.

---
## Act 2: Data-Driven Gate Candidates

We test 3 candidate gates. For each method × cluster, the gate produces a single number. We then look for a universal cutoff that separates "method works" from "method doesn't work."

- **Gate A: Median Pearson** — median correlation of top-20 genes with the pattern
- **Gate B: Gene Spread** — average std of top-20 genes across timepoints (lower = tighter band)
- **Gate C: Centroid Fit** — MSE between the mean of top-20 gene curves and the pattern

In [ ]:
# Compute all three gate scores for every method × cluster combination

def compute_gate_scores(n=20):
    """Compute Gate A (median Pearson), Gate B (gene spread), Gate C (centroid fit)
    for each method × cluster. Returns three DataFrames."""
    
    gate_a = pd.DataFrame(index=method_names, columns=cluster_names, dtype=float)  # median Pearson
    gate_b = pd.DataFrame(index=method_names, columns=cluster_names, dtype=float)  # gene spread
    gate_c = pd.DataFrame(index=method_names, columns=cluster_names, dtype=float)  # centroid MSE
    
    for method_name in method_names:
        for cluster_name in cluster_names:
            top_genes = get_top_genes(method_name, cluster_name, n)
            gene_df = gene_dataLT[cluster_name]
            pattern = np.array(constraint_patternsLT[cluster_name]['constraint'])
            
            # Get expression matrix for top genes
            valid_genes = [g for g in top_genes if g in gene_df.index]
            gene_matrix = gene_df.loc[valid_genes].values  # shape: (n_genes, 4)
            
            # Gate A: Median Pearson correlation with pattern
            correlations = []
            for row in gene_matrix:
                if np.std(row) == 0 or np.std(pattern) == 0:
                    correlations.append(0.0)
                else:
                    r, _ = scipy_stats.pearsonr(row, pattern)
                    correlations.append(r if np.isfinite(r) else 0.0)
            gate_a.loc[method_name, cluster_name] = np.median(correlations)
            
            # Gate B: Average std across timepoints (spread of top genes)
            if len(gene_matrix) > 1:
                std_per_timepoint = np.std(gene_matrix, axis=0)  # std across genes at each timepoint
                gate_b.loc[method_name, cluster_name] = np.mean(std_per_timepoint)
            else:
                gate_b.loc[method_name, cluster_name] = 0.0
            
            # Gate C: Centroid MSE (MSE between mean of top genes and pattern)
            centroid = np.mean(gene_matrix, axis=0)
            gate_c.loc[method_name, cluster_name] = np.mean((centroid - pattern) ** 2)
    
    return gate_a, gate_b, gate_c

gate_a, gate_b, gate_c = compute_gate_scores(n=20)

print("Gate A — Median Pearson (higher = genes follow pattern shape):")
print(gate_a.round(3))
print()
print("Gate B — Gene Spread (lower = tighter band):")
print(gate_b.round(4))
print()
print("Gate C — Centroid MSE (lower = centroid matches pattern):")
print(gate_c.round(6))

In [ ]:
# Visualize all three gates as heatmaps side by side

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
gate_labels = ['Gate A: Median Pearson\n(higher = better)', 
               'Gate B: Gene Spread\n(lower = better)', 
               'Gate C: Centroid MSE\n(lower = better)']
gates = [gate_a, gate_b, gate_c]
cmaps = ['RdYlGn', 'RdYlGn_r', 'RdYlGn_r']  # reversed for "lower is better" gates

for ax, gate, label, cmap in zip(axes, gates, gate_labels, cmaps):
    data = gate.astype(float).values
    im = ax.imshow(data, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(cluster_names)))
    ax.set_xticklabels([c.replace('LT', '') for c in cluster_names], fontsize=9)
    ax.set_yticks(range(len(method_names)))
    ax.set_yticklabels(method_names, fontsize=9)
    ax.set_title(label, fontsize=11, fontweight='bold')
    
    # Annotate cells
    for i in range(len(method_names)):
        for j in range(len(cluster_names)):
            val = data[i, j]
            fmt = f'{val:.3f}' if abs(val) > 0.001 else f'{val:.1e}'
            ax.text(j, i, fmt, ha='center', va='center', fontsize=8,
                    color='white' if abs(val) > np.mean(data) else 'black')
    
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Gate Candidate Scores: 6 Methods × 4 Clusters', fontsize=13)
plt.tight_layout()
plt.savefig('plots/act2_gate_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Find natural gaps: sort all 24 scores for each gate and look for a clear separation

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
gate_configs = [
    ('Gate A: Median Pearson', gate_a, False, 'Higher = method works'),  # higher is better
    ('Gate B: Gene Spread', gate_b, True, 'Lower = method works'),       # lower is better
    ('Gate C: Centroid MSE', gate_c, True, 'Lower = method works'),      # lower is better
]

for ax, (title, gate, ascending, note) in zip(axes, gate_configs):
    # Flatten to list of (score, label) pairs
    entries = []
    for method_name in method_names:
        for cluster_name in cluster_names:
            score = float(gate.loc[method_name, cluster_name])
            short_cluster = cluster_name.replace('LT', '')
            entries.append((score, f'{method_name}\n{short_cluster}'))
    
    # Sort
    entries.sort(key=lambda x: x[0], reverse=not ascending)
    scores_sorted = [e[0] for e in entries]
    labels_sorted = [e[1] for e in entries]
    
    # Color by score quality
    colors = []
    for s in scores_sorted:
        if ascending:
            colors.append('#2ecc71' if s < np.median(scores_sorted) else '#e74c3c')
        else:
            colors.append('#2ecc71' if s > np.median(scores_sorted) else '#e74c3c')
    
    bars = ax.barh(range(len(scores_sorted)), scores_sorted, color=colors, alpha=0.7)
    ax.set_yticks(range(len(labels_sorted)))
    ax.set_yticklabels(labels_sorted, fontsize=6)
    ax.set_title(f'{title}\n({note})', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    
    # Find largest gap
    gaps = [abs(scores_sorted[i+1] - scores_sorted[i]) for i in range(len(scores_sorted)-1)]
    max_gap_idx = np.argmax(gaps)
    gap_threshold = (scores_sorted[max_gap_idx] + scores_sorted[max_gap_idx + 1]) / 2
    ax.axvline(x=gap_threshold, color='red', linestyle='--', linewidth=2, 
               label=f'Natural gap = {gap_threshold:.4f}')
    ax.legend(fontsize=8)

plt.suptitle('All 24 Gate Scores Sorted — Looking for Natural Cutoff Gaps', fontsize=12)
plt.tight_layout()
plt.savefig('plots/act2_gate_gaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare gates: for each candidate, show the active/silenced grid at the natural gap cutoff
# and verify it matches visual judgment

def get_active_grid(gate, threshold, higher_is_better=True):
    """Return DataFrame of ✓/✗ based on gate scores and threshold."""
    grid = pd.DataFrame(index=method_names, columns=cluster_names, dtype=str)
    for m in method_names:
        for c in cluster_names:
            val = float(gate.loc[m, c])
            if higher_is_better:
                grid.loc[m, c] = '✓' if val >= threshold else '✗'
            else:
                grid.loc[m, c] = '✓' if val <= threshold else '✗'
    return grid

# Compute natural gap thresholds for each gate
def find_gap_threshold(gate, ascending=True):
    """Find threshold at the largest gap in sorted scores."""
    scores = gate.astype(float).values.flatten()
    scores_sorted = np.sort(scores) if ascending else np.sort(scores)[::-1]
    gaps = [abs(scores_sorted[i+1] - scores_sorted[i]) for i in range(len(scores_sorted)-1)]
    max_gap_idx = np.argmax(gaps)
    return (scores_sorted[max_gap_idx] + scores_sorted[max_gap_idx + 1]) / 2

thresh_a = find_gap_threshold(gate_a, ascending=False)  # higher is better, sort descending
thresh_b = find_gap_threshold(gate_b, ascending=True)    # lower is better
thresh_c = find_gap_threshold(gate_c, ascending=True)    # lower is better

print(f"Gate A threshold (Median Pearson ≥ {thresh_a:.4f}):")
grid_a = get_active_grid(gate_a, thresh_a, higher_is_better=True)
print(grid_a)
print()
print(f"Gate B threshold (Gene Spread ≤ {thresh_b:.4f}):")
grid_b = get_active_grid(gate_b, thresh_b, higher_is_better=False)
print(grid_b)
print()
print(f"Gate C threshold (Centroid MSE ≤ {thresh_c:.6f}):")
grid_c = get_active_grid(gate_c, thresh_c, higher_is_better=False)
print(grid_c)
print()

# Count active methods per cluster for each gate
print("Active methods per cluster:")
for label, grid in [('Gate A', grid_a), ('Gate B', grid_b), ('Gate C', grid_c)]:
    counts = {c: (grid[c] == '✓').sum() for c in cluster_names}
    print(f"  {label}: {dict(counts)}")

### Sensitivity Analysis

Try multiple thresholds for each gate to see how robust the results are. We want the "right answer" to be stable — not dependent on picking exactly the right number.

In [ ]:
# Sensitivity: how does the active/silenced grid change as we move the threshold?

def count_active(gate, threshold, higher_is_better=True):
    """Return total active method-cluster pairs."""
    data = gate.astype(float).values.flatten()
    if higher_is_better:
        return (data >= threshold).sum()
    else:
        return (data <= threshold).sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Gate A: sweep Pearson threshold
thresholds_a = np.arange(-0.5, 1.0, 0.05)
for cluster_name in cluster_names:
    active_counts = []
    for t in thresholds_a:
        col = gate_a[cluster_name].astype(float)
        active_counts.append((col >= t).sum())
    short = cluster_name.replace('LT', '')
    axes[0].plot(thresholds_a, active_counts, label=short, linewidth=2)
axes[0].axvline(x=thresh_a, color='red', linestyle='--', label=f'Gap = {thresh_a:.2f}')
axes[0].set_xlabel('Pearson threshold')
axes[0].set_ylabel('Active methods')
axes[0].set_title('Gate A: Median Pearson')
axes[0].legend(fontsize=7)
axes[0].set_ylim(-0.5, 6.5)

# Gate B: sweep spread threshold
thresholds_b = np.linspace(0, gate_b.astype(float).values.max() * 1.1, 50)
for cluster_name in cluster_names:
    active_counts = []
    for t in thresholds_b:
        col = gate_b[cluster_name].astype(float)
        active_counts.append((col <= t).sum())
    short = cluster_name.replace('LT', '')
    axes[1].plot(thresholds_b, active_counts, label=short, linewidth=2)
axes[1].axvline(x=thresh_b, color='red', linestyle='--', label=f'Gap = {thresh_b:.4f}')
axes[1].set_xlabel('Spread threshold')
axes[1].set_ylabel('Active methods')
axes[1].set_title('Gate B: Gene Spread')
axes[1].legend(fontsize=7)
axes[1].set_ylim(-0.5, 6.5)

# Gate C: sweep centroid MSE threshold
thresholds_c = np.linspace(0, gate_c.astype(float).values.max() * 1.1, 50)
for cluster_name in cluster_names:
    active_counts = []
    for t in thresholds_c:
        col = gate_c[cluster_name].astype(float)
        active_counts.append((col <= t).sum())
    short = cluster_name.replace('LT', '')
    axes[2].plot(thresholds_c, active_counts, label=short, linewidth=2)
axes[2].axvline(x=thresh_c, color='red', linestyle='--', label=f'Gap = {thresh_c:.6f}')
axes[2].set_xlabel('Centroid MSE threshold')
axes[2].set_ylabel('Active methods')
axes[2].set_title('Gate C: Centroid MSE')
axes[2].legend(fontsize=7)
axes[2].set_ylim(-0.5, 6.5)

plt.suptitle('Sensitivity: Active Methods per Cluster vs Threshold', fontsize=12)
plt.tight_layout()
plt.savefig('plots/act2_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

### Gated Voting

Apply the best gate to the ensemble voting. Only active methods contribute votes per cluster. We run this for all three gates so we can compare the final consensus gene lists.

In [ ]:
def gated_ensemble(active_grid, n=20):
    """Run ensemble voting with only active methods voting per cluster.
    Returns dict of cluster -> DataFrame with gene, votes, avg_rank, active_methods."""
    results = {}
    
    for cluster_name in cluster_names:
        # Get active methods for this cluster
        active_methods = [m for m in method_names if active_grid.loc[m, cluster_name] == '✓']
        
        if len(active_methods) == 0:
            results[cluster_name] = pd.DataFrame({
                'votes': [], 'avg_rank': [], 'active_methods': []
            })
            continue
        
        # Compute ranks only for active methods
        rank_df = pd.DataFrame()
        for mname in active_methods:
            scores, ascending = all_methodsLT[mname]
            s = scores[cluster_name]['constraint']
            rank_df[mname] = s.rank(ascending=ascending, method='min')
        
        votes = (rank_df <= n).sum(axis=1)
        avg_rank = rank_df.mean(axis=1)
        
        result = pd.DataFrame({
            'votes': votes,
            'avg_rank': avg_rank,
        })
        result = result.sort_values(['votes', 'avg_rank'], ascending=[False, True])
        result['active_methods'] = len(active_methods)
        results[cluster_name] = result
    
    return results

# Run gated ensemble for all three gates
results_a = gated_ensemble(grid_a)
results_b = gated_ensemble(grid_b)
results_c = gated_ensemble(grid_c)

# Also run ungated for comparison
ungated_grid = pd.DataFrame('✓', index=method_names, columns=cluster_names)
results_ungated = gated_ensemble(ungated_grid)

# Print summary
for label, res, grid in [('Ungated', results_ungated, ungated_grid),
                          ('Gate A (Median Pearson)', results_a, grid_a), 
                          ('Gate B (Gene Spread)', results_b, grid_b),
                          ('Gate C (Centroid MSE)', results_c, grid_c)]:
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    for cluster_name in cluster_names:
        df = res[cluster_name]
        if len(df) == 0:
            print(f"\n  {cluster_name}: NO ACTIVE METHODS")
            continue
        active = [m for m in method_names if grid.loc[m, cluster_name] == '✓']
        top10 = df.head(10)
        max_v = top10['active_methods'].iloc[0] if len(top10) > 0 else 0
        print(f"\n  {cluster_name} ({len(active)}/{len(method_names)} methods active: {', '.join(active)})")
        print(f"  {'Rank':<5} {'Gene':<20} {'Votes':>6} {'Avg Rank':>10}")
        print(f"  {'-'*45}")
        for i, (gene, row) in enumerate(top10.iterrows(), 1):
            print(f"  {i:<5} {gene:<20} {int(row['votes']):>3}/{max_v:<3} {row['avg_rank']:>10.1f}")

In [ ]:
# Consensus gene plots: for each gate, plot top-20 genes per cluster with opacity = consensus

def plot_gated_ensemble(results, gate_label, active_grid, save_prefix=''):
    """Plot ensemble gene overlays for each cluster with gated voting."""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    for idx, cluster_name in enumerate(cluster_names):
        ax = axes[idx]
        gene_df = gene_dataLT[cluster_name]
        pattern_vals = constraint_patternsLT[cluster_name]['constraint']
        
        df = results[cluster_name]
        active = [m for m in method_names if active_grid.loc[m, cluster_name] == '✓']
        
        if len(df) == 0 or len(active) == 0:
            ax.text(0.5, 0.5, 'No active\nmethods', transform=ax.transAxes,
                    ha='center', va='center', fontsize=14, color='red')
            ax.set_title(f'{cluster_name.replace("LT", "")}', fontsize=10)
            continue
        
        top20 = df.head(20)
        max_votes = int(top20['active_methods'].iloc[0])
        
        for gene, row in top20.iterrows():
            if gene in gene_df.index:
                v = int(row['votes'])
                alpha = 0.2 + 0.6 * (v / max_votes) if max_votes > 0 else 0.4
                ax.plot(x, gene_df.loc[gene], color='steelblue', alpha=alpha, linewidth=1)
        
        ax.plot(x, pattern_vals, color='crimson', linewidth=2.5, linestyle='--')
        ax.set_xlabel('Timepoint')
        ax.set_xticks(x)
        short_cluster = cluster_name.replace('LT', '')
        ax.set_title(f'{short_cluster}\n({len(active)} methods active)', fontsize=10)
        if idx == 0:
            ax.set_ylabel('Gene counts')
    
    plt.suptitle(f'{gate_label} — Top 20 Consensus Genes per Cluster\n'
                 f'(opacity ∝ vote fraction)', fontsize=12)
    plt.tight_layout()
    if save_prefix:
        plt.savefig(f'plots/{save_prefix}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_gated_ensemble(results_ungated, 'Ungated (all 6 methods)', ungated_grid, 'ensemble_ungated')
plot_gated_ensemble(results_a, f'Gate A: Median Pearson ≥ {thresh_a:.2f}', grid_a, 'ensemble_gate_a')
plot_gated_ensemble(results_b, f'Gate B: Gene Spread ≤ {thresh_b:.4f}', grid_b, 'ensemble_gate_b')
plot_gated_ensemble(results_c, f'Gate C: Centroid MSE ≤ {thresh_c:.6f}', grid_c, 'ensemble_gate_c')

In [ ]:
# Overlap comparison: how many top-20 genes are shared between gated and ungated ensembles?

print("Top-20 Overlap: Gated vs Ungated Ensemble")
print("=" * 60)
for cluster_name in cluster_names:
    ungated_top20 = set(results_ungated[cluster_name].head(20).index)
    short = cluster_name.replace('LT', '')
    print(f"\n{short}:")
    for label, res in [('Gate A (Pearson)', results_a),
                       ('Gate B (Spread)', results_b),
                       ('Gate C (Centroid)', results_c)]:
        gated_top20 = set(res[cluster_name].head(20).index) if len(res[cluster_name]) > 0 else set()
        overlap = len(ungated_top20 & gated_top20)
        print(f"  {label}: {overlap}/20 shared with ungated")

# Also compare gates with each other
print(f"\n\nTop-20 Overlap Between Gates")
print("=" * 60)
for cluster_name in cluster_names:
    short = cluster_name.replace('LT', '')
    top_a = set(results_a[cluster_name].head(20).index) if len(results_a[cluster_name]) > 0 else set()
    top_b = set(results_b[cluster_name].head(20).index) if len(results_b[cluster_name]) > 0 else set()
    top_c = set(results_c[cluster_name].head(20).index) if len(results_c[cluster_name]) > 0 else set()
    print(f"\n{short}:")
    print(f"  A vs B: {len(top_a & top_b)}/20")
    print(f"  A vs C: {len(top_a & top_c)}/20")
    print(f"  B vs C: {len(top_b & top_c)}/20")

## Summary & Literature Notes

**Gate candidates tested:**
- **Gate A (Median Pearson):** For each method × cluster, computes the median Pearson correlation between the method's top-20 genes and the constraint pattern. Captures shape agreement.
- **Gate B (Gene Spread):** Average standard deviation of the top-20 genes across timepoints. Captures band tightness but can be fooled by tight wrong clusters.
- **Gate C (Centroid MSE):** MSE between the mean of the top-20 gene curves and the constraint pattern. Combines shape and proximity — centroid won't match if genes are scattered OR clustered around the wrong shape.

**Literature justification:**
- Pearson correlation thresholds (|r| > 0.5–0.7) are standard in gene co-expression network analysis (Zhang & Horvath, 2005; Langfelder & Horvath, 2008 — WGCNA)
- Centroid-based cluster quality evaluation is widely used in unsupervised learning (Rousseeuw, 1987 — Silhouettes; Dunn, 1973 — Dunn Index)
- Ensemble methods with quality-weighted inputs outperform equal-weight ensembles (Kolde et al., 2012 — Robust Rank Aggregation)
- DTW significance via permutation testing validates the use of data-driven cutoffs over fixed thresholds (Cavill et al., 2013 — DTW4Omics)

### Observations from the Gate Comparison

**Gate A (Median Pearson)** is too permissive — almost everything passes (threshold ≈ 0.29). Even MSE on clusterThree gets median r = 1.0 because the top genes follow the right *direction* (dip at t=3) even at wildly different magnitudes. Pearson is scale-invariant, so it can't detect magnitude mismatch — exactly the concern raised earlier.

**Gate B (Gene Spread)** partially works — silences Cosine on clusterOne and clusterFour, sMAPE on clusterFour. But doesn't catch MSE on clusterThree because the genes, while far from the pattern, are tightly grouped.

**Gate C (Centroid MSE)** has the right idea — centroid far from pattern = bad. But it fails on clusterOne because ALL methods have high centroid MSE there (the pattern's absolute values are just larger). The fix: **normalize by the pattern's range** so scores are comparable across clusters.

### Normalized Gate C: Relative Centroid Error

In [ ]:
# Gate C-norm: Centroid MSE normalized by pattern range squared
# This makes scores comparable across clusters with different absolute scales

gate_c_norm = pd.DataFrame(index=method_names, columns=cluster_names, dtype=float)

for method_name in method_names:
    for cluster_name in cluster_names:
        top_genes = get_top_genes(method_name, cluster_name, n=20)
        gene_df = gene_dataLT[cluster_name]
        pattern = np.array(constraint_patternsLT[cluster_name]['constraint'])
        
        valid_genes = [g for g in top_genes if g in gene_df.index]
        gene_matrix = gene_df.loc[valid_genes].values
        
        centroid = np.mean(gene_matrix, axis=0)
        raw_mse = np.mean((centroid - pattern) ** 2)
        
        # Normalize by pattern range squared (+ eps to avoid division by zero)
        pattern_range = pattern.max() - pattern.min()
        eps = 1e-10
        gate_c_norm.loc[method_name, cluster_name] = raw_mse / (pattern_range ** 2 + eps)

print("Gate C-norm — Relative Centroid Error (lower = centroid matches pattern):")
print(gate_c_norm.astype(float).round(3))
print()

# Find gap and show grid
thresh_c_norm = find_gap_threshold(gate_c_norm, ascending=True)
grid_c_norm = get_active_grid(gate_c_norm, thresh_c_norm, higher_is_better=False)
print(f"Threshold (Relative Centroid Error ≤ {thresh_c_norm:.4f}):")
print(grid_c_norm)
print()
counts = {c.replace('LT', ''): (grid_c_norm[c] == '✓').sum() for c in cluster_names}
print(f"Active methods per cluster: {counts}")

# Sorted bar chart for this gate
fig, ax = plt.subplots(figsize=(8, 8))
entries = []
for method_name in method_names:
    for cluster_name in cluster_names:
        score = float(gate_c_norm.loc[method_name, cluster_name])
        short_cluster = cluster_name.replace('LT', '')
        entries.append((score, f'{method_name} / {short_cluster}'))

entries.sort(key=lambda x: x[0])
scores_sorted = [e[0] for e in entries]
labels_sorted = [e[1] for e in entries]
colors = ['#2ecc71' if s <= thresh_c_norm else '#e74c3c' for s in scores_sorted]

ax.barh(range(len(scores_sorted)), scores_sorted, color=colors, alpha=0.7)
ax.set_yticks(range(len(labels_sorted)))
ax.set_yticklabels(labels_sorted, fontsize=7)
ax.axvline(x=thresh_c_norm, color='red', linestyle='--', linewidth=2, 
           label=f'Cutoff = {thresh_c_norm:.3f}')
ax.set_title('Gate C-norm: Relative Centroid Error\n(green = active, red = silenced)', fontweight='bold')
ax.set_xlabel('Relative Centroid Error')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('plots/act2_gate_c_norm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Run gated ensemble with the normalized Gate C and plot results
results_c_norm = gated_ensemble(grid_c_norm)

plot_gated_ensemble(results_c_norm, f'Gate C-norm: Relative Centroid Error ≤ {thresh_c_norm:.3f}', 
                    grid_c_norm, 'ensemble_gate_c_norm')

# Print the final rankings
print(f"\nGate C-norm Rankings")
print(f"{'='*60}")
for cluster_name in cluster_names:
    df = results_c_norm[cluster_name]
    if len(df) == 0:
        print(f"\n  {cluster_name}: NO ACTIVE METHODS")
        continue
    active = [m for m in method_names if grid_c_norm.loc[m, cluster_name] == '✓']
    top10 = df.head(10)
    max_v = int(top10['active_methods'].iloc[0])
    short = cluster_name.replace('LT', '')
    print(f"\n  {short} ({len(active)}/{len(method_names)} active: {', '.join(active)})")
    print(f"  {'Rank':<5} {'Gene':<20} {'Votes':>6} {'Avg Rank':>10}")
    print(f"  {'-'*45}")
    for i, (gene, row) in enumerate(top10.iterrows(), 1):
        print(f"  {i:<5} {gene:<20} {int(row['votes']):>3}/{max_v:<3} {row['avg_rank']:>10.1f}")

---
## Act 3: Raw-Space MAE — "Does this line look like that line?"

Instead of normalizing and then applying fancy metrics, just compute the Mean Absolute Error between each gene's raw LT expression and the constraint pattern. No normalization, no tricks — the metric that most closely matches what the human eye sees on a plot.

In [ ]:
# Raw-space MAE: for each gene, compute mean absolute error vs the pattern
# in the actual LT expression values (no normalization)

raw_mae_results = {}

for cluster_name in cluster_names:
    pattern = np.array(constraint_patternsLT[cluster_name]['constraint'])
    gene_df = gene_dataLT[cluster_name]
    
    # Vectorized MAE computation
    gene_matrix = gene_df.values  # (17856, 4)
    mae_per_gene = np.mean(np.abs(gene_matrix - pattern), axis=1)
    
    raw_mae_results[cluster_name] = pd.Series(mae_per_gene, index=gene_df.index)

# Show top 20 genes per cluster
print("Raw-Space MAE — Top 20 Genes (lowest MAE = closest to pattern on the plot)")
print("=" * 70)
for cluster_name in cluster_names:
    short = cluster_name.replace('LT', '')
    top20 = raw_mae_results[cluster_name].nsmallest(20)
    print(f"\n{short}:")
    print(f"  {'Rank':<5} {'Gene':<20} {'MAE':>10}")
    print(f"  {'-'*35}")
    for i, (gene, mae) in enumerate(top20.items(), 1):
        print(f"  {i:<5} {gene:<20} {mae:>10.6f}")

In [ ]:
# Plot raw MAE top-20 genes per cluster — this should look like what the eye picks

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, cluster_name in enumerate(cluster_names):
    ax = axes[idx]
    gene_df = gene_dataLT[cluster_name]
    pattern_vals = constraint_patternsLT[cluster_name]['constraint']
    
    top20 = raw_mae_results[cluster_name].nsmallest(20)
    
    for gene in top20.index:
        if gene in gene_df.index:
            ax.plot(x, gene_df.loc[gene], color='steelblue', alpha=0.5, linewidth=1)
    
    ax.plot(x, pattern_vals, color='crimson', linewidth=2.5, linestyle='--')
    ax.set_xlabel('Timepoint')
    ax.set_xticks(x)
    short = cluster_name.replace('LT', '')
    best_mae = top20.iloc[0]
    ax.set_title(f'{short}\n(best MAE = {best_mae:.4f})', fontsize=10)
    if idx == 0:
        ax.set_ylabel('Gene counts')
    
    legend_elements = [
        Line2D([0], [0], color='crimson', linewidth=2.5, linestyle='--', label='Constraint pattern'),
        Line2D([0], [0], color='steelblue', alpha=0.5, label=f'Top 20 genes (raw MAE)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=7)

plt.suptitle('Raw-Space MAE — Top 20 Genes per Cluster\n'
             '"Which genes literally sit on the pattern line?"', fontsize=12)
plt.tight_layout()
plt.savefig('plots/act3_raw_mae.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare raw MAE top-20 with each of the 6 original methods' top-20

print("Overlap: Raw MAE top-20 vs each method's top-20")
print("=" * 60)
for cluster_name in cluster_names:
    short = cluster_name.replace('LT', '')
    mae_top20 = set(raw_mae_results[cluster_name].nsmallest(20).index)
    print(f"\n{short}:")
    for method_name in method_names:
        method_top20 = set(get_top_genes(method_name, cluster_name, 20))
        overlap = len(mae_top20 & method_top20)
        print(f"  vs {method_name:<10}: {overlap}/20 shared")

In [ ]:
# Generate individual per-cluster plots for slides

# Gated ensemble (Gate C-norm)
for cluster_name in cluster_names:
    fig, ax = plt.subplots(figsize=(10, 6))
    gene_df = gene_dataLT[cluster_name]
    pattern_vals = constraint_patternsLT[cluster_name]['constraint']
    df = results_c_norm[cluster_name]
    active = [m for m in method_names if grid_c_norm.loc[m, cluster_name] == '✓']
    
    if len(df) == 0 or len(active) == 0:
        ax.text(0.5, 0.5, 'No active methods', transform=ax.transAxes,
                ha='center', va='center', fontsize=18, color='red')
    else:
        top20 = df.head(20)
        max_votes = int(top20['active_methods'].iloc[0])
        for gene, row in top20.iterrows():
            if gene in gene_df.index:
                v = int(row['votes'])
                alpha = 0.2 + 0.6 * (v / max_votes) if max_votes > 0 else 0.4
                ax.plot(x, gene_df.loc[gene], color='steelblue', alpha=alpha, linewidth=1.5)
        ax.plot(x, pattern_vals, color='crimson', linewidth=3, linestyle='--')
        legend_elements = [
            Line2D([0], [0], color='crimson', linewidth=3, linestyle='--', label='Constraint pattern'),
            Line2D([0], [0], color='steelblue', alpha=0.6, linewidth=1.5, label=f'Top 20 genes'),
        ]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=11)
    
    short = cluster_name.replace('LT', '')
    ax.set_title(f'{short} — Gated Ensemble ({len(active)}/6 methods active)', fontsize=14)
    ax.set_xlabel('Timepoint', fontsize=12)
    ax.set_ylabel('Gene counts', fontsize=12)
    ax.set_xticks(x)
    plt.tight_layout()
    plt.savefig(f'plots/gated_{short}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved plots/gated_{short}.png')

# Raw MAE
for cluster_name in cluster_names:
    fig, ax = plt.subplots(figsize=(10, 6))
    gene_df = gene_dataLT[cluster_name]
    pattern_vals = constraint_patternsLT[cluster_name]['constraint']
    top20 = raw_mae_results[cluster_name].nsmallest(20)
    for gene in top20.index:
        if gene in gene_df.index:
            ax.plot(x, gene_df.loc[gene], color='steelblue', alpha=0.5, linewidth=1.5)
    ax.plot(x, pattern_vals, color='crimson', linewidth=3, linestyle='--')
    short = cluster_name.replace('LT', '')
    best_mae = top20.iloc[0]
    ax.set_title(f'{short} — Raw MAE (best = {best_mae:.4f})', fontsize=14)
    ax.set_xlabel('Timepoint', fontsize=12)
    ax.set_ylabel('Gene counts', fontsize=12)
    ax.set_xticks(x)
    legend_elements = [
        Line2D([0], [0], color='crimson', linewidth=3, linestyle='--', label='Constraint pattern'),
        Line2D([0], [0], color='steelblue', alpha=0.5, linewidth=1.5, label=f'Top 20 genes (raw MAE)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'plots/rawmae_{short}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved plots/rawmae_{short}.png')

In [ ]:
# Visual: Log transform vs Normalization — why one preserves distance and the other doesn't
# Using exaggerated values so the difference is obvious

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pattern near zero, one gene close, one gene VERY far
pattern =    [5, 1, 4, 3]
gene_close = [7, 2, 5, 4]       # close to pattern
gene_far =   [500, 100, 400, 300]  # same shape, 100x larger

# --- Panel 1: Raw values ---
ax = axes[0]
ax.plot(x, pattern, color='crimson', linewidth=3, linestyle='--', marker='D', markersize=8, label='Pattern')
ax.plot(x, gene_close, color='#2ecc71', linewidth=2.5, marker='o', markersize=8, label='Gene A (close)')
ax.plot(x, gene_far, color='#3498db', linewidth=2.5, marker='s', markersize=8, label='Gene B (100× larger)')
ax.set_title('Raw Expression Values', fontsize=13, fontweight='bold')
ax.set_xlabel('Timepoint', fontsize=11)
ax.set_ylabel('Expression', fontsize=11)
ax.set_xticks(x)
ax.legend(fontsize=9, loc='upper left')
ax.annotate('Gene A hugs the pattern\nGene B is 100× away', xy=(6, 400), xytext=(6.5, 250),
            fontsize=10, ha='center', fontweight='bold')

# --- Panel 2: After log transform ---
ax2 = axes[1]
log_pattern = [np.log1p(v) for v in pattern]
log_close = [np.log1p(v) for v in gene_close]
log_far = [np.log1p(v) for v in gene_far]

ax2.plot(x, log_pattern, color='crimson', linewidth=3, linestyle='--', marker='D', markersize=8, label='Pattern')
ax2.plot(x, log_close, color='#2ecc71', linewidth=2.5, marker='o', markersize=8, label='Gene A (close)')
ax2.plot(x, log_far, color='#3498db', linewidth=2.5, marker='s', markersize=8, label='Gene B (100× larger)')
ax2.set_title('After Log Transform', fontsize=13, fontweight='bold')
ax2.set_xlabel('Timepoint', fontsize=11)
ax2.set_ylabel('log(1 + Expression)', fontsize=11)
ax2.set_xticks(x)
ax2.legend(fontsize=9, loc='upper left')
ax2.annotate('Compressed, but\nGene A is still closer', xy=(6, 1.6), xytext=(6.5, 3.5),
            fontsize=10, ha='center', color='#2ecc71', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=1.5))

# --- Panel 3: After min-max normalization to (0,1] ---
ax3 = axes[2]
def minmax_norm(vals):
    eps = 1e-10
    mn, mx = min(vals), max(vals)
    return [(v - mn + eps) / (mx - mn + eps) for v in vals]

norm_pattern = minmax_norm(pattern)
norm_close = minmax_norm(gene_close)
norm_far = minmax_norm(gene_far)

ax3.plot(x, norm_pattern, color='crimson', linewidth=3, linestyle='--', marker='D', markersize=8, label='Pattern')
ax3.plot(x, norm_close, color='#2ecc71', linewidth=2.5, marker='o', markersize=8, label='Gene A (close)')
ax3.plot(x, norm_far, color='#3498db', linewidth=2.5, marker='s', markersize=8, label='Gene B (100× larger)')
ax3.set_title('After Min-Max Normalization', fontsize=13, fontweight='bold')
ax3.set_xlabel('Timepoint', fontsize=11)
ax3.set_ylabel('Normalized to (0, 1]', fontsize=11)
ax3.set_xticks(x)
ax3.legend(fontsize=9, loc='lower right')
ax3.annotate('All three are identical!\n100× difference erased', xy=(3, 0.25), xytext=(5, 0.4),
            fontsize=11, ha='center', color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

plt.suptitle('Log transform compresses but preserves distance — Normalization erases distance entirely',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('plots/log_vs_normalization.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Generate visual for "Why Pearson = 1.0 doesn't mean visual match"
# Wide aspect ratio to fit a 16:9 beamer slide

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

pattern = [0.04, 0.0, 0.037, 0.035]
gene_a = [0.04, 0.0, 0.037, 0.035]   # sits on pattern
gene_b = [1.50, 0.0, 1.30, 1.20]     # same shape, 37x larger

# Left: both genes plotted with the pattern
ax = axes[0]
ax.plot(x, pattern, color='crimson', linewidth=3, linestyle='--', label='Constraint pattern', zorder=5)
ax.plot(x, gene_a, color='#2ecc71', linewidth=2.5, marker='o', markersize=8, label='Gene A (r = 1.0, MAE = 0.0)')
ax.plot(x, gene_b, color='#3498db', linewidth=2.5, marker='s', markersize=8, label='Gene B (r = 1.0, MAE = 0.87)')
ax.set_xlabel('Timepoint', fontsize=11)
ax.set_ylabel('Gene Expression (LT)', fontsize=11)
ax.set_xticks(x)
ax.set_title('Both genes have Pearson r = 1.0', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(-0.1, 1.7)

ax.annotate('37× larger\nbut "perfect" correlation', 
            xy=(3, 0.0), xytext=(5.5, 0.7),
            fontsize=10, ha='center',
            arrowprops=dict(arrowstyle='->', color='#3498db', lw=2),
            color='#3498db', fontweight='bold')

# Right: zoomed in to show Gene A actually sits on pattern
ax2 = axes[1]
ax2.plot(x, pattern, color='crimson', linewidth=3, linestyle='--', label='Constraint pattern', zorder=5)
ax2.plot(x, gene_a, color='#2ecc71', linewidth=2.5, marker='o', markersize=8, label='Gene A — sits on pattern')
ax2.set_xlabel('Timepoint', fontsize=11)
ax2.set_ylabel('Gene Expression (LT)', fontsize=11)
ax2.set_xticks(x)
ax2.set_title('Zoomed in: only Gene A matches', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9, loc='upper right')
ax2.set_ylim(-0.005, 0.055)

plt.tight_layout()
plt.savefig('plots/pearson_visual_explanation.png', dpi=150, bbox_inches='tight')
plt.show()